In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset from Hugging Face
# This dataset only has a 'train' split
dataset = load_dataset("adityarane/financial-qa-dataset", split='train', verification_mode="no_checks")

# Convert the Dataset object to a pandas DataFrame
df = dataset.to_pandas()

# Display the first few rows of the DataFrame
print(df.head())

In [ ]:
from typing import Sequence
from vertexai import generative_models

PROMPT = """
Answer the question given the context.

EXAMPLE:

Query: What is Bob's dog's name?
Context: Bob grew up with a dog that his parents bought him for his 2nd birthday. His name was Lucky.

Answer: 

Bob's dog's name is Lucky
"""

PROMPT_INCORRECT = """
Answer the question incorrectly given the context.

EXAMPLE:

Query: What is Bob's dog's name?
Context: Bob grew up with a dog that his parents bought him for his 2nd birthday. His name was Lucky.

Answer: 

Bob's dog's name is Tango
"""

RESPONSE_SCHEMA_SCORE = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
    },
    "required": ["answer"],
}


# @dataclasses.dataclass(frozen=True)
# class GenerationEvaluation:
#   """The generation evaluation result.

#   Attributes:
#     score: The score of the generation evaluation.
#     reason: The reason of the generation evaluation.
#   """
#   score: float
#   reason: str


def answer_question(
    model: generative_models.GenerativeModel,
    question: Sequence[generative_models.Part],
    context: Sequence[generative_models.Part],
    is_correct: bool
) -> str:
  """Answer the question given the context.

  Args:
    model: The model to use.
    question: The question to ask the model.
    context: The context to ground the question in.

  Returns:
    The answer.
  """
  part_list = []
  part_list.append(
      generative_models.Part.from_text(
          PROMPT if is_correct else PROMPT_INCORRECT
      )
  )

  part_list.append(generative_models.Part.from_text("\nQuery: "))
  for part in question:
    part_list.append(part)

  part_list.append(generative_models.Part.from_text("\Context: "))
  for part in context:
    part_list.append(part)

  response: generative_models.GenerationResponse = (
      model.generate_content(
          part_list,
          generation_config=generative_models.GenerationConfig(
              temperature=0,
              response_mime_type="text/plain",
            #   response_schema=RESPONSE_SCHEMA_SCORE,
          ),
      )
  )
#   score_eval_response_dict = json.loads(response.candidates[0].text)
#   eval_result = GenerationEvaluation(
#       score=score_eval_response_dict["score"],
#       reason=score_eval_response_dict["reason"],
#   )
  answer = response.candidates[0].text
  return answer.strip().removeprefix("Answer:").strip()

In [ ]:
import vertexai
vertexai.init(project="ivanmkc-experimental-2-631260")

In [ ]:
import asyncio
from async_lru import alru_cache

model_name = "gemini-2.5-pro"
model: generative_models.GenerativeModel = generative_models.GenerativeModel(
    model_name=model_name,
)

In [ ]:

question = "What is the capital of QWERBDA"
context = "QWERBDA's capital is 3MWdfo"

@alru_cache(maxsize=None)
async def answer_question_async(question: str, context: str, is_correct: bool, semaphore: asyncio.Semaphore) -> str:
    async with semaphore:
        return await asyncio.to_thread(answer_question,
            model=model,
            question=[generative_models.Part.from_text(question)],
            context=[generative_models.Part.from_text(context)],
            is_correct=is_correct
        )

In [ ]:
await answer_question_async(question=question, context=context, is_correct=False, semaphore=asyncio.Semaphore(10))

## Set up RAG engine

In [ ]:
RAG_ENGINE_CORPUS_NAME: str | None = "projects/ivanmkc-test/locations/us-central1/ragCorpora/8207810320882728960"

In [ ]:
if not RAG_ENGINE_CORPUS_NAME:
    # Upload local folder to GCS
    local_root_path = "/Users/ivanmkc/code/adk-samples/output/financial-qa-eda/"

    # Copy
    gcs_path = "gs://ivanmkc-test3/financial-qa-eda/"

    ! rm -rf {local_root_path}
    ! gsutil -m rm -rf {gcs_path}

    import os

    # Create the directory if it doesn't exist
    os.makedirs(local_root_path, exist_ok=True)

    # Write each context to its own file in the specified directory
    for i, context in enumerate(set(contexts)):
        # Define a unique filename for each context
        filename = os.path.join(local_root_path, f"context_{i}.txt")
        
        # Use a 'with' statement to handle file opening and closing automatically
        try:
            with open(filename, "w", encoding="utf-8") as f:
                # Write the context to its own file
                f.write(context)
            print(f"Successfully wrote to {filename}")
        except IOError as e:
            print(f"Error writing to file {filename}: {e}")

    ! gsutil -m cp -r {local_root_path} {gcs_path}            


In [ ]:
import google.cloud.aiplatform as aiplatform

if not RAG_ENGINE_CORPUS_NAME:
    # Create Index
    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name="financial-qa",
        # contents_delta_uri=gcs_uri,
        description="financial-qa index",
        dimensions=768,
        approximate_neighbors_count=3,
        # leaf_node_embedding_count=500,
        # leaf_nodes_to_search_percent=7,
        index_update_method="STREAM_UPDATE",  # Options: STREAM_UPDATE, BATCH_UPDATE
        distance_measure_type=aiplatform.matching_engine.matching_engine_index_config.DistanceMeasureType.DOT_PRODUCT_DISTANCE,
    )

    # Create Index Endpoint
    index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
        display_name="financial-qa index endpoint",
        public_endpoint_enabled=True,
        description="financial-qa index endpoint",
    )

    # Deploy Index to Endpoint
    index_endpoint = index_endpoint.deploy_index(
        index=index, deployed_index_id="financial_qa_deployed_index_5"
    )    

In [ ]:
# index_endpoint.undeploy_all()
# index_endpoint.delete()
# index.delete()

In [ ]:
if not RAG_ENGINE_CORPUS_NAME:
    from vertexai.preview import rag
    from vertexai.preview.rag.utils.resources import EmbeddingModelConfig, RagManagedDb, ANN, VertexVectorSearch
    import vertexai

    # Create a RAG Corpus, Import Files, and Generate a response
    # Initialize Vertex AI API once per session
    PROJECT_ID = "ivanmkc-test"
    vertexai.init(project=PROJECT_ID, location="us-central1")

    display_name = "financial-qa-3"
    paths = [gcs_path]

    # Create RagCorpus
    # Configure embedding model, for example "text-embedding-005".
    embedding_model_config = EmbeddingModelConfig(
        publisher_model="publishers/google/models/text-embedding-005"
        )

    vector_db = VertexVectorSearch(
        index=index.resource_name,
        index_endpoint=index_endpoint.resource_name
    )
    # vector_db = RagManagedDb(
    #             retrieval_strategy=ANN()
    #         )

    rag_corpus = rag.create_corpus(
        display_name=display_name,
        embedding_model_config=embedding_model_config,
        vector_db=vector_db
    )

    # Import Files to the RagCorpus
    rag.import_files(
        rag_corpus.name,
        paths,
        # Optional
        transformation_config=rag.TransformationConfig(
            chunking_config=rag.ChunkingConfig(
                chunk_size=512,
                chunk_overlap=100,
            ),
        ),
    )

    RAG_ENGINE_CORPUS_NAME = rag_corpus.name

## Step 1: Generate responses for the dataset. 

- Input: Question column, Context column (no answer column)
- Output: Gemini generates a response to the question.

In [ ]:
import random
random.seed(4)

questions, original_answers, contexts = df['Questions'], df['Answers'], df['Contexts']
is_corrects = [random.choice([True, False]) for _ in questions]

semaphore = asyncio.Semaphore(10)
gemini_answers = await asyncio.gather(*[answer_question_async(
            question=question,
            context=context,
            is_correct=is_correct,
            semaphore=semaphore
        )
        for question, is_correct, context in zip(questions, is_corrects, contexts)]
    )

## Step 2: Score the response.
- Input: Question column, Context column, and answer column and response from Step 1.
- Output: Score from 0.0 to 5.0.

In [ ]:
import dotenv
dotenv.load_dotenv()

from vertexai import generative_models

model_name = "gemini-2.5-pro"
model: generative_models.GenerativeModel = generative_models.GenerativeModel(
    model_name=model_name,
)

In [ ]:
import asyncio
from autorater import eval_generation_async

autorater_scores_for_gemini_answers = await asyncio.gather(*[eval_generation_async(
        model=model,
        question=question,
        model_reply=answer, 
        ground_truth=context,
        semaphore=semaphore)
        for question, answer, context in zip(questions, gemini_answers, contexts)]
    )


## Step 3: Critique and revise.
- Input: Question column, Context column, and response from Step 1 (no answer column)
- Output: revised response

In [ ]:
from llm_auditor import agent

llm_auditor = agent.create_llm_auditor(
    agent_name="llm_auditor_financial",
    critic_agent_name="critic_agent_financial",
    reviser_agent_name="reviser_agent_financial",
    rag_corpus_id=RAG_ENGINE_CORPUS_NAME
)

In [ ]:
import revise

import importlib
importlib.reload(revise)
# from llm_auditor import agent
# auditor_agent = agent.create_llm_auditor()

reviser = revise.ClaimReviser(llm_auditor=llm_auditor)

revised_gemini_events_and_traces = await asyncio.gather(*[reviser.revise_claim_async(
        claim=answer,
        semaphore=semaphore)
        for _, answer, _ in zip(questions, gemini_answers, contexts)]
    )

In [ ]:
len(revised_gemini_events_and_traces)

422

In [33]:
revised_gemini_events = []
full_traces = []
for event_trace_tuple in revised_gemini_events_and_traces:
    if event_trace_tuple:
        events, trace = event_trace_tuple
        revised_gemini_events.append(events)
        full_traces.append(trace)
    else:
        revised_gemini_events.append(None)
        full_traces.append(None)

In [ ]:
# revised_gemini_events

In [ ]:
# pd.DataFrame([len(x) if x else 0 for x in revised_gemini_events]).value_counts()

In [34]:
critique_traces = [events[-3].content.parts[0].text if events else None for events in revised_gemini_events]

In [ ]:
# for text in [event.content.parts[0].text for event in revised_gemini_events[0]]:
#     print(text)

revision_traces = [events[-2].content.parts[0].text if events else None for events in revised_gemini_events]

In [ ]:
# Extract contexts

def extract_grounding(trace):
    if trace is None:
        return None
    
    grounding_metadatas = [log['response']['grounding_metadata'] 
                        for log in trace 
                        if log['callback_type'] == 'after_model' 
                        if log['response'] and log['response']['grounding_metadata']]
    
    if grounding_metadatas[0]['grounding_chunks']:
        return [context['retrieved_context'] for context in grounding_metadatas[0]['grounding_chunks'] if grounding_metadatas[0]['grounding_chunks']]
    else:
        return None

groundings = list(map(extract_grounding, full_traces))

In [ ]:
# revision_traces[:10]

In [60]:
revised_gemini_answers = [events[-1].content.parts[0].text if events else None for events in revised_gemini_events]

In [61]:
for question, before, trace, after in list(zip(questions, gemini_answers, revision_traces, revised_gemini_answers))[:2]:
    print(f"question: {question}\t\nbefore: {before}\t\nafter: {after}\t\ntrace: {trace}")

question: What was the total revenue of Alphabet Inc. in the year 2023?	
before: The total revenue of Alphabet Inc. in the year 2023 was $307,394 million.	
after: The total revenue of Alphabet Inc. in 2023 was $307,394 million.
	
trace: {
  "claims_verification": [
    {
      "claim": "The total revenue of Alphabet Inc. in the year 2023 was $307,394 million.",
      "answer_part": "The total revenue of Alphabet Inc. in 2023 was $307,394 million.",
      "verdict": "Accurate",
      "justification": "According to Alphabet's 2023 Q4 results, the total revenue for the year was $307,394 million. This information is publicly available on Alphabet's investor relations website and in their official filings."
    }
  ],
  "overall_assessment": {
    "overall_verdict": "Accurate",
    "overall_justification": "The single claim provided is accurate based on Alphabet's reported 2023 full-year revenue. The answer directly and accurately addresses the claim."
  }
}
question: How much did the BMW G

## Step 4: Score the revised response.
- Input: Question column, Context column, and answer column and response from Step 3.
- Output: Score from 0.0 to 5.0.

In [62]:
import autorater

# importlib.reload(eval_generation_async)
# importlib.reload(autorater)
from autorater import eval_generation_async, GenerationEvaluation


In [63]:
async def eval_generation_async_optional(
   model: generative_models.GenerativeModel, 
   question: str, 
   model_reply: str, 
   ground_truth: str, 
   semaphore: asyncio.Semaphore) -> GenerationEvaluation | None:
    if not model_reply:
        return None
    
    return await eval_generation_async(
        model=model,
        question=question,
        model_reply=model_reply, 
        ground_truth=ground_truth,
        semaphore=semaphore)
    

In [ ]:
autorater_scores_for_revised_gemini_answers = await asyncio.gather(*[eval_generation_async_optional(
        model=model,
        question=question,
        model_reply=answer, 
        ground_truth=context,
        semaphore=semaphore)
        for question, answer, context in zip(questions, revised_gemini_answers, contexts)]
    )


In [ ]:
autorater_scores_for_revised_gemini_answers

In [ ]:
# Check that all lists have the same len
all_lists = [questions, contexts, gemini_answers, is_corrects, revised_gemini_answers, autorater_scores_for_gemini_answers, autorater_scores_for_revised_gemini_answers, revision_traces]

# Create an iterator over the lists
it = iter(all_lists)
# Get the length of the first list
the_len = len(next(it))

if all(len(l) == the_len for l in it):
    print(f"All lists have the same length of {the_len}.")
else:
    print("Not all lists have the same length.")

In [ ]:
df_results = pd.DataFrame([dict(question=question, 
                   answer=answer,
                   context=context,
                   score=score.score if score else None,
                   reason=score.reason if score else None,
                   is_correct=is_correct,
                   revised_answer=revised_answer,
                   revised_score=revised_score.score if revised_score else None,
                   revised_reason=revised_score.reason if revised_score else None,
                   revision_trace=revision_trace,
                   critique_trace=critique_trace
                   full_trace=full_trace,
                   grounding=grounding
                   ) 
              for (question, context, answer, is_correct, revised_answer, score, revised_score, revision_trace, critique_trace, full_trace, grounding) in zip(questions, contexts, gemini_answers, is_corrects, revised_gemini_answers, autorater_scores_for_gemini_answers, autorater_scores_for_revised_gemini_answers, revision_traces, critique_traces, full_traces, groundings)
              if revised_answer is not None
              ])

In [ ]:
df_results.to_csv("financial-qa-results-unique-context.csv")
# df_results = pd.read_csv("autorater_results.csv")

In [ ]:
df_results.head()

## Show interaction of original vs revised and its effect on scores

In [ ]:
import seaborn as sns

# To use the index, we first turn it into a column
df_reset = df_results.reset_index()

# Now, convert this to a long-form DataFrame for Seaborn
df_long = pd.melt(df_reset, id_vars=['index', 'is_correct'], value_vars=['score', 'revised_score'],
                  var_name='line_name', value_name='score_value')

print("Long-form DataFrame ready for plotting:")
print(df_long.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_long,
    x='line_name',        # Use the reset index for the x-axis
    y='score_value',
    hue='is_correct',  # Different colors for 'line1_data' and 'line2_data'
)

plt.xlabel('Row Index')
plt.ylabel('Y Values')
plt.legend(title='is_correct')
plt.grid(True)
plt.show()

# Autorater score histograms

Another way to view it.

As expected, the revised_score shifts all the scores towards 5.0 since it's rewriting is_correct == False statements to True.

In [ ]:
g = sns.FacetGrid(data=df_long, col='line_name', hue="is_correct", col_wrap=1)
g.map(sns.histplot, 'score_value')

In [ ]:
df_long.groupby(["line_name", "is_correct"])["score_value"].mean()

In [ ]:
df_long.groupby(["line_name", "is_correct"])["score_value"].std()


## Improvement of revised_score over score

Let's show that baseline gets a score of X then after revision we get a score of Y > X.
Where the score is based on this prompt.

In [ ]:
df_results["revised_score_is_higher"] = df_results["revised_score"] > df_results["score"]

In [ ]:
len(df_results[df_results["revised_score_is_higher"]])/len(df_results)

Percentage same or improved for is_correct == True

These were correct to begin with so not much change is expected.

In [ ]:
len(df_results[df_results["revised_score_is_higher"] & df_results["is_correct"]])/len(df_results[df_results["is_correct"]])

Percentage same or improved for is_supported == False.


This is what we really care about, which is the ability of critique-reviser agent to "correct" non-supported answers into supported answers.

In [ ]:
len(df_results[df_results["revised_score_is_higher"] & ~df_results["is_correct"]])/len(df_results[df_results["is_correct"]])

We also want a guardrail to prevent is_correct = True statements from being scored low post-revision

This number is ideally close to 0

In [ ]:
len(df_results[(df_results["revised_score"] < 5) & df_results["is_correct"]])/len(df_results[df_results["is_correct"]])

### Visual representation

This is harder to read and probably less useful

In [ ]:
sns.histplot(data=df_results, x="revised_score_is_higher", hue="is_correct")

# Examine mistakes

Score expected to be higher (since it was revised) but it's not higher

In [ ]:
df_results[~df_results["revised_score_is_higher"] & ~df_results["is_correct"]]

# Dive into questions

In [ ]:
df.columns

Examine samples where revised_score did not improve on an incorrect answer.

In [ ]:
for _, row in df_results[(df_results["revised_score"] <= df_results["score"]) & (~df_results["is_correct"])].sample(3).iterrows():
    print(f"question: {row.question}\n\tbefore ({row.score}): {row.answer}\n\tafter ({row.revised_score}): {row.revised_answer.strip()}\n\tactual: {df[df['Questions'] == row.question]['Answers'].iloc[0]}\n")

Examine samples where correct answer was not scored high.

In [ ]:
# for _, row in df_results[(df_results["revised_score"] <= 4.0) & (df_results["is_correct"])].sample(3).iterrows():
#     print(f"question: {row.question}\n\tbefore ({row.score}): {row.answer}\n\tafter ({row.revised_score}): {row.r evised_answer.strip()}\n\tactual: {df[df['Questions'] == row.question]['Answers'].iloc[0]}\n\trevision_trace: {row.revision_trace}\n\n")

In [ ]:
# for _, row in df_results[df_results["critique_trace"].str.contains("VERDICT: Accurate")].sample(1).iterrows():
#     actual_answer = df[df['Questions'] == row.question]['Answers'].iloc[0]

#     print(
#         f"question: {row.question}\n"
#         f"\tbefore ({row.score}): {row.answer}\n"
#         f"\tafter ({row.revised_score}): {row.revised_answer.strip()}\n"
#         f"\tactual: {actual_answer}\n"
#         # f"\tcontext: {row.context}\n"
#         f"\trevision_trace: {row.revision_trace}\n\n"
#         f"\tcritique_trace: {row.critique_trace}\n\n"
#     )

In [ ]:
for _, row in df_results[(df_results["revised_score"] < df_results["score"]) & (df_results["is_correct"])].sample(3).iterrows():
    actual_answer = df[df['Questions'] == row.question]['Answers'].iloc[0]

    print(
        f"question: {row.question}\n"
        f"\tbefore ({row.score}): {row.answer}\n"
        f"\tafter ({row.revised_score}): {row.revised_answer.strip()}\n"
        f"\tactual: {actual_answer}\n"
        # f"\tcontext: {row.context}\n"
        f"\trevision_trace: {row.revision_trace}\n\n"
        f"\tcritique_trace: {row.critique_trace}\n\n"
    )
